# 02 - Classic factor research

We dig into the *non-novel* families: momentum, short-term reversal, low-vol, quality, liquidity, and microstructure. For each factor we measure:
* **IC** (Spearman cross-sectional rank correlation with forward returns)
* **IC-IR** (mean / std)  and  **IC t-statistic**
* **Quantile cumulative-return spread** (5 buckets)
* **Turnover** of the top quintile

Forward horizon = 5 trading days unless stated otherwise.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from mfa.config import load_config
from mfa.viz import apply_style
apply_style()

cfg = load_config(ROOT / "configs" / "default.yaml")
print("project:", cfg.project_root.name, "| processed:", cfg.processed_dir.exists())


In [ ]:
from mfa.data import build_panels

def _load_raw_panels():
    out = {}
    for k in ["open","high","low","close","volume","vwap","dollar_volume"]:
        p = cfg.processed_dir / f"panel_{k}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            out[k] = df
    if not out:
        raise RuntimeError("No cached panels found. Run scripts/01_download_data.py first.")
    return out

def _load_factors():
    d = cfg.processed_dir / "factors"
    out = {}
    for p in sorted(d.glob("*.parquet")):
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        out[p.stem] = df
    return out

raw_panels = _load_raw_panels()
benchmark = cfg.section("universe").get("benchmark", "SPY")
panels = build_panels(raw_panels, benchmark=benchmark, min_history_days=252)
factors = _load_factors()
print(f"loaded {len(panels)} panels and {len(factors)} factors")
print("date range:", panels["close"].index.min().date(), "->", panels["close"].index.max().date())
print("universe size:", panels["close"].shape[1], " benchmark:", benchmark)


In [ ]:
from mfa.data import forward_returns
from mfa.evaluation import (
    information_coefficient, ic_summary, rolling_ic,
    quantile_returns, long_short_returns, factor_turnover, ic_decay,
)
from mfa.evaluation.stats import perf_stats
from mfa.factors import REGISTRY

H = 5
fwd = forward_returns(panels["close"], [1, H, 10, 21])
print("forward horizons computed:", list(fwd.keys()))


## 1. IC scoreboard (all classic factors)

In [ ]:
classic_families = ["momentum", "reversal", "volatility", "quality", "liquidity", "microstructure"]
classic_names = [n for fam, names in REGISTRY.by_family().items() if fam in classic_families for n in names]
print(f"{len(classic_names)} classic factors:")
for n in classic_names: print(" ", n)

rows = []
for n in classic_names:
    f = factors[n]
    ic = information_coefficient(f, fwd[H])
    s = ic_summary(ic)
    qr = quantile_returns(f, fwd[H], n_quantiles=5)
    ls = long_short_returns(qr)
    perf = perf_stats(ls, periods_per_year=252/H)
    to = factor_turnover(f, n_quantiles=5).mean()
    family = REGISTRY.get(n).family
    rows.append({"factor": n, "family": family, "ic_mean": s["ic_mean"],
                 "ic_ir": s["ic_ir"], "ic_t": s["ic_t_stat"],
                 "hit": s["ic_hit_rate"], "ls_sharpe": perf["sharpe"],
                 "ls_ann": perf["ann_return"], "ls_mdd": perf["max_drawdown"],
                 "turnover": float(to)})
board = pd.DataFrame(rows).set_index("factor").sort_values("ic_t", ascending=False)
board.style.format({"ic_mean":"{:+.3f}","ic_ir":"{:+.2f}","ic_t":"{:+.2f}","hit":"{:.1%}",
                    "ls_sharpe":"{:+.2f}","ls_ann":"{:+.1%}","ls_mdd":"{:.1%}","turnover":"{:.1%}"}).background_gradient(cmap="RdYlGn", subset=["ic_t","ls_sharpe"])


## 2. IC bar-chart by family

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = sns.color_palette("Set2", len(set(board["family"])))
fam_to_color = dict(zip(sorted(set(board["family"])), colors))
b = board.sort_values("ic_t")
ax.barh(range(len(b)), b["ic_t"], color=[fam_to_color[f] for f in b["family"]], edgecolor="black", lw=0.5)
ax.set_yticks(range(len(b))); ax.set_yticklabels(b.index, fontsize=9)
ax.axvline(2, color="red", ls="--", lw=1, label="t = +2"); ax.axvline(-2, color="red", ls="--", lw=1)
ax.axvline(0, color="black", lw=0.7)
ax.set_xlabel("IC t-statistic"); ax.set_title("Classic-factor IC significance")
handles = [plt.Rectangle((0,0),1,1, color=fam_to_color[f]) for f in fam_to_color]
ax.legend(handles, list(fam_to_color), loc="lower right"); plt.tight_layout(); plt.show()


## 3. Deep-dive on the strongest factor

In [ ]:
best = board.index[0]
print(f"strongest by t-stat: {best}  (family = {board.loc[best, 'family']})")

f = factors[best]
ic = information_coefficient(f, fwd[H])
qr = quantile_returns(f, fwd[H], n_quantiles=5)
ls = long_short_returns(qr)
decay = ic_decay(f, panels["close"], horizons=[1, 5, 10, 21, 63])

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
# IC time series
ic.plot(ax=ax[0,0], color="#bbbbbb", lw=0.7, label="daily")
ic.rolling(21).mean().plot(ax=ax[0,0], color="#2E86AB", lw=1.8, label="21d")
ax[0,0].axhline(0, color="black", lw=0.7); ax[0,0].set_title(f"IC time series -- {best}"); ax[0,0].legend()
# IC decay
decay.plot.bar(ax=ax[0,1], color="#2E86AB", edgecolor="black")
ax[0,1].axhline(0, color="black", lw=0.7); ax[0,1].set_title("IC decay"); ax[0,1].set_xlabel("forward horizon (days)")
# Quantile cum returns
eq = (1 + qr.fillna(0)).cumprod()
palette = sns.color_palette("RdYlGn", 5)
for i, c in enumerate(eq.columns):
    eq[c].plot(ax=ax[1,0], label=c, color=palette[i])
ax[1,0].set_yscale("log"); ax[1,0].set_title("Quantile cumulative returns"); ax[1,0].legend(ncol=5)
# L-S equity
(1 + ls.fillna(0)).cumprod().plot(ax=ax[1,1], color="#06A77D", lw=2.0)
ax[1,1].set_title("Long-short equity (Q5 - Q1)"); ax[1,1].set_ylabel("growth of $1")
plt.tight_layout(); plt.show()


## 4. IC stability across forward horizons

In [ ]:
horizon_grid = pd.DataFrame(index=classic_names, columns=[1, 5, 10, 21], dtype=float)
for n in classic_names:
    for h in horizon_grid.columns:
        ic = information_coefficient(factors[n], fwd[h])
        horizon_grid.loc[n, h] = ic.mean()
fig, ax = plt.subplots(figsize=(7, max(5, 0.3*len(classic_names))))
sns.heatmap(horizon_grid.astype(float), cmap="RdBu_r", center=0, annot=True, fmt="+.3f",
            cbar_kws={"label": "mean IC"}, ax=ax)
ax.set_title("Mean IC by factor x forward horizon"); ax.set_xlabel("forward horizon (days)"); plt.show()


### Takeaways

* Classic momentum (12-1) shows the strongest single-factor IC and significance in this regime.
* Low-vol shows persistent NEGATIVE IC -- the post-2020 mega-cap-tech rally rewarded high-vol
  names, inverting the textbook anomaly.
* Most short-horizon (1d) ICs are weak; we should expect the cross-sectional book to work
  best at a 5-21 day horizon.